# Langchain AI Agent with OpenRouter LLM Notebook

Welcome to my notebook! You can find more content on my YouTube channel: [NextNexaOfficial](https://www.youtube.com/@NextNexaOfficial)

## Setup and Dependencies
This section handles the installation of necessary libraries and environment setup.

NEW

In [ ]:
!pip install langchain langchain-openrouter python-dotenv requests
!pip install langchain-community langchain-text-splitters langchain-huggingface sentence-transformers



This cell installs all the required Python libraries for the project, including `langchain`, `langchain-openrouter`, `python-dotenv`, `requests`, `langchain-community`, `langchain-text-splitters`, `langchain-huggingface`, and `sentence-transformers`. These libraries are crucial for building the RAG (Retrieval Augmented Generation) system, interacting with LLMs, and handling text processing.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

This cell sets up the environment by retrieving the `OPENROUTER_API_KEY` from Colab's user data secrets and setting it as an environment variable. This key is necessary for authenticating with the OpenRouter API to use their language models.

In [ ]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="openai/gpt-4o-mini"
)


Here, the `ChatOpenRouter` class is imported and initialized with a specific language model, `openai/gpt-4o-mini`. This `llm` object will be used to interact with the chosen LLM for generating responses.

In [ ]:
try:
    test_message = "Hello, how are you?"
    test_response = llm.invoke(test_message)
    print(f"Model test successful: {test_response.content[:50]}...")
except Exception as e:
    print(f"Model test failed: {e}")

This cell performs a quick test to ensure that the initialized language model (`llm`) is working correctly. It sends a simple message and prints a part of the model's response or an error message if the test fails.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []


This cell imports `HumanMessage` and `AIMessage` from `langchain_core.messages`, which are used to represent conversational turns. It also initializes an empty `chat_history` list to store the conversation flow.

In [ ]:
import requests
from langchain.tools import tool

@tool
def web_search(query: str):
    """Search the web using DuckDuckGo."""
    r = requests.get(
        "https://api.duckduckgo.com/",
        params={"q": query, "format": "json"}
    )
    data = r.json()

    abstract = data.get("Abstract")
    heading = data.get("Heading")
    related = data.get("RelatedTopics")

    if abstract:
        return abstract

    if heading:
        return f"No abstract found. DuckDuckGo heading: {heading}"

    if related:
        return f"No abstract found. Related topics exist for '{query}'."

    return f"No results found for '{query}'."




This cell defines a `web_search` tool using `DuckDuckGo`'s API. This tool allows the LLM to perform web searches and retrieve summaries or related topics based on a given query.

## Tools Definition
This section defines various tools that the LLM can use to perform specific actions, such as web searching, file creation, and web scraping.

In [ ]:
@tool
def create_file_from_url(url: str):
    """Download content from a URL and save it into the Colab files folder."""
    import requests
    import os

    folder = "/content/files"
    os.makedirs(folder, exist_ok=True)

    filename = url.split("/")[-1] or "downloaded_file.txt"
    filepath = os.path.join(folder, filename)

    response = requests.get(url)
    if response.status_code != 200:
        return f"Failed to download: {response.status_code}"

    with open(filepath, "wb") as f:
        f.write(response.content)

    return f"File saved at: {filepath}"



This cell defines a `create_file_from_url` tool. It enables the LLM to download content from a specified URL and save it as a file in the `/content/files` directory within the Colab environment.

In [ ]:
@tool
def scrape_and_save(url: str):
    """Scrape text content from a webpage and save it into /content/files/."""
    import requests
    from bs4 import BeautifulSoup
    import os

    # Create folder
    folder = "/content/files"
    os.makedirs(folder, exist_ok=True)

    # Fetch page
    try:
        response = requests.get(url)
    except Exception as e:
        return f"Error fetching URL: {e}"

    if response.status_code != 200:
        return f"Failed to fetch page: {response.status_code}"

    # Parse HTML
    soup = BeautifulSoup(response.text, "html.parser")

    # Extract readable text
    text = soup.get_text(separator="\n")

    # Save file
    filename = url.replace("https://", "").replace("http://", "").replace("/", "_") + ".txt"
    filepath = os.path.join(folder, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)

    return f"Scraped text saved at: {filepath}"


This cell defines a `scrape_and_save` tool. This tool fetches the content of a webpage, extracts readable text using `BeautifulSoup`, and saves the text into a `.txt` file in the `/content/files` directory.

In [ ]:
tools = [web_search, create_file_from_url, scrape_and_save]



This cell creates a list named `tools` that aggregates all the defined tools (`web_search`, `create_file_from_url`, `scrape_and_save`). This list will be used by the LLM to bind these capabilities to its conversational chain.

In [ ]:
from google.colab import files

uploaded = files.upload()

filename = list(uploaded.keys())[0]
print("Uploaded:", filename)


This cell allows the user to upload a local file to the Colab environment using `files.upload()`. The uploaded file's name is then stored in the `filename` variable, which will be used for further processing.

## Document Upload and RAG Setup
This section handles uploading local documents and setting up the Retrieval Augmented Generation (RAG) system.

In [ ]:
!pip install faiss-cpu


This cell installs the `faiss-cpu` library, which is a library for efficient similarity search and clustering of dense vectors. It is used here to create a FAISS vector store for fast retrieval of document chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Load uploaded document
with open(filename, "r", encoding="utf-8") as f:
    text = f.read()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(text)

# Embeddings
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Vector store
db = FAISS.from_texts(chunks, emb)

def rag_search(query):
    docs = db.similarity_search(query, k=3)
    return "\n\n".join([d.page_content for d in docs])


This cell sets up the RAG (Retrieval Augmented Generation) mechanism. It loads the previously uploaded document, splits it into smaller chunks using `RecursiveCharacterTextSplitter`, creates embeddings for these chunks using `HuggingFaceEmbeddings`, and then builds a FAISS vector store. Finally, it defines a `rag_search` function to query this vector store for relevant document chunks.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. "
     "When the user asks anything about searching the web, ALWAYS call the tool named web_search. "
     "User must type: 'web_search: <query>' or 'search web: <query>' or 'search web for <query>'."),
    ("placeholder", "{history}"),
    ("human", "{input}")
])


parser = StrOutputParser()

chain = prompt | llm.bind_tools(tools) | parser



This cell defines the `ChatPromptTemplate` that guides the LLM's behavior, including system instructions and placeholders for chat history and user input. It also initializes a `StrOutputParser` to parse the LLM's output and constructs the main `chain` by combining the prompt, the LLM with bound tools, and the parser.

## Conversational Chain
This section defines the prompt template and the overall conversational chain for the LLM.

In [ ]:
def chat(message):
    global chat_history

    # 1. Manual RAG trigger
    if message.lower().startswith("search document"):
        query = message.replace("search document", "").strip()
        return rag_search(query)

    # 2. Manual web_search trigger (this is the FIX)
    if (
        message.lower().startswith("web search") or
        message.lower().startswith("web_search") or
        message.lower().startswith("search web")
    ):
        query = (
            message.lower()
            .replace("web search", "")
            .replace("web_search", "")
            .replace("search web", "")
            .replace("for", "")
            .strip()
        )
        return web_search.run(query)

    # 3. Manual trigger for file creation tool
    if message.lower().startswith("create_file_from_url"):
        url = message.replace("create_file_from_url", "").strip()
        return create_file_from_url.run(url)

    # 3. Manual trigger for web scraping tool
    if message.lower().startswith("scrape"):
        url = message.replace("scrape", "").strip()
        return scrape_and_save.run(url)



    # 3. Format memory
    formatted_history = []
    for msg in chat_history:
        if msg["role"] == "user":
            formatted_history.append(HumanMessage(content=msg["content"]))
        else:
            formatted_history.append(AIMessage(content=msg["content"]))

    # 4. Run LLM with tools + memory
    response = chain.invoke({
        "history": formatted_history,
        "input": message
    })

    # 5. Save memory
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "ai", "content": response})

    return response




This cell defines the `chat` function, which orchestrates the conversational flow. It includes logic for manually triggering RAG search, web search, file creation, and web scraping tools based on user input. It also manages the chat history and invokes the LLM chain to generate responses.

## Chat Functionality
This section implements the core chat logic, including manual tool triggering and interaction with the LLM.

In [ ]:
print("Chat started. Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    reply = chat(user_input)
    print("AI:", reply, "\n")


This cell initiates the interactive chat loop. It continuously prompts the user for input, calls the `chat` function to get a response from the AI, and prints the conversation. The chat continues until the user types 'exit'.